# Device Results Visualization

Select a results directory, device, and simulation CSV.
Then filter using input-variable dropdowns and inspect interactive measured vs. simulation plots.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import clear_output, display


In [1]:
%pip install plotly
%pip install -U nbformat


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 646.8 kB/s eta 0:00:00 kB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:

# Single default candidate (relative)
_default_results_dir = Path("../models_results")
RESULTS_DIR = _default_results_dir

run_dir_input = widgets.Text(
    value=str(_default_results_dir),
    description="Run Dir:",
    placeholder="Enter path to models_results",
    layout=widgets.Layout(width="900px"),
    style={"description_width": "80px"},
)
apply_run_dir_btn = widgets.Button(description="Apply", button_style="primary", icon="check")
run_dir_status = widgets.Output()


def _apply_run_dir(_=None):
    global RESULTS_DIR
    with run_dir_status:
        clear_output(wait=True)
        candidate = Path(run_dir_input.value).expanduser().resolve()

        if not candidate.exists():
            print(f"Results directory does not exist: {candidate}")
            return
        if not candidate.is_dir():
            print(f"Path is not a directory: {candidate}")
            return

        RESULTS_DIR = candidate
        print(f"Using results directory: {RESULTS_DIR}")


apply_run_dir_btn.on_click(_apply_run_dir)
_apply_run_dir()

display(widgets.VBox([
    widgets.HBox([run_dir_input, apply_run_dir_btn]),
    run_dir_status,
]))



In [5]:


COLORS = {
    "range_fill": "rgba(204, 194, 194, 0.65)",
    "typ": "#2F2F2F",
    "sim_alt": "#9A9A9A",
    "meas": "#1f77b4",
    "outside": "#d62728",
}
ALL_VALUE = "__ALL__"


def split_sim_columns(df, out_first):
    prefix = f"{out_first}_sim_"
    sim_cols = [c for c in df.columns if c.startswith(prefix)]
    if not sim_cols:
        return [], None, []

    typ_candidates = [c for c in sim_cols if c.split("_")[-1].lower() in {"typ", "tt", "typical"}]
    typ_col = typ_candidates[0] if typ_candidates else None
    other_cols = [c for c in sim_cols if c != typ_col]
    return sim_cols, typ_col, other_cols


def outside_mask(y, lo, hi):
    return (y < lo) | (y > hi)


def format_value(v):
    if isinstance(v, (int, float, np.integer, np.floating)) and pd.notna(v):
        return f"{float(v):.6g}"
    return str(v)


def make_group_title(panel_df, block_id, context_vars):
    row = panel_df.iloc[0]
    context = [f"{v}={format_value(row[v])}" for v in context_vars if v in panel_df.columns]
    context_str = " | ".join(context)
    if context_str:
        return context_str
    return f"Block {block_id}"


def plot_grouped_blocks(filtered_df, output_vars, sweep_var, context_vars):
    if "block_id" not in filtered_df.columns:
        print("Column 'block_id' is missing in CSV.")
        return

    block_ids = list(pd.unique(filtered_df["block_id"]))
    if not block_ids:
        print("No blocks to plot after filtering.")
        return

    for bid in block_ids:
        panel_df = filtered_df[filtered_df["block_id"] == bid].copy()
        if panel_df.empty:
            continue

        panel_outputs = parse_output_vars(panel_df) or output_vars
        if not panel_outputs:
            continue

        ncols = min(2, len(panel_outputs))
        nrows = math.ceil(len(panel_outputs) / ncols)
        fig = make_subplots(
            rows=nrows,
            cols=ncols,
            subplot_titles=[out_var.upper() for out_var in panel_outputs],
            horizontal_spacing=0.08,
            vertical_spacing=0.12,
        )
        legend_seen = set()

        for i, out_var in enumerate(panel_outputs):
            row_idx = (i // ncols) + 1
            col_idx = (i % ncols) + 1
            block_sorted = panel_df.sort_values(sweep_var)

            x = pd.to_numeric(block_sorted[sweep_var], errors="coerce").to_numpy(dtype=float)
            if np.isnan(x).all():
                x = np.arange(len(block_sorted), dtype=float)

            meas_col = f"{out_var}_meas"
            sim_cols, typ_col, other_cols = split_sim_columns(block_sorted, out_var)

            y_lo = y_hi = None
            if other_cols:
                sims = block_sorted[other_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
                y_lo = np.full(len(block_sorted), np.nan, dtype=float)
                y_hi = np.full(len(block_sorted), np.nan, dtype=float)
                valid_rows = np.any(np.isfinite(sims), axis=1)
                if np.any(valid_rows):
                    y_lo[valid_rows] = np.nanmin(sims[valid_rows], axis=1)
                    y_hi[valid_rows] = np.nanmax(sims[valid_rows], axis=1)
                    upper_name = "Upper Bound"
                    fig.add_trace(
                        go.Scatter(
                            x=x,
                            y=y_hi,
                            mode="lines",
                            line=dict(color="#9E9E9E", width=1.2),
                            name=upper_name,
                            legendgroup=upper_name,
                            showlegend=upper_name not in legend_seen,
                            hovertemplate="%{x}<br>Upper: %{y}<extra></extra>",
                        ),
                        row=row_idx,
                        col=col_idx,
                    )
                    legend_seen.add(upper_name)
                    range_name = "Simulation Range"
                    fig.add_trace(
                        go.Scatter(
                            x=x,
                            y=y_lo,
                            mode="lines",
                            line=dict(width=0),
                            fill="tonexty",
                            fillcolor=COLORS["range_fill"],
                            name=range_name,
                            legendgroup=range_name,
                            showlegend=range_name not in legend_seen,
                            hovertemplate="%{x}<br>Low: %{y}<extra></extra>",
                        ),
                        row=row_idx,
                        col=col_idx,
                    )
                    legend_seen.add(range_name)

            if typ_col is not None:
                typ_name = typ_col.replace(f"{out_var}_sim_", "").upper()
                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=pd.to_numeric(block_sorted[typ_col], errors="coerce").to_numpy(dtype=float),
                        mode="lines",
                        line=dict(color=COLORS["typ"], dash="dash", width=2),
                        name=typ_name,
                        legendgroup=typ_name,
                        showlegend=typ_name not in legend_seen,
                    ),
                    row=row_idx,
                    col=col_idx,
                )
                legend_seen.add(typ_name)
            else:
                for c in sim_cols:
                    sim_name = c.replace(f"{out_var}_sim_", "").upper()
                    fig.add_trace(
                        go.Scatter(
                            x=x,
                            y=pd.to_numeric(block_sorted[c], errors="coerce").to_numpy(dtype=float),
                            mode="lines",
                            line=dict(color=COLORS["sim_alt"], dash="dash", width=1.4),
                            opacity=0.8,
                            name=sim_name,
                            legendgroup=sim_name,
                            showlegend=sim_name not in legend_seen,
                        ),
                        row=row_idx,
                        col=col_idx,
                    )
                    legend_seen.add(sim_name)

            if meas_col in block_sorted.columns:
                meas_name = "Measured"
                y_meas = pd.to_numeric(block_sorted[meas_col], errors="coerce").to_numpy(dtype=float)
                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=y_meas,
                        mode="lines+markers",
                        line=dict(color=COLORS["meas"], width=2),
                        marker=dict(size=5),
                        name=meas_name,
                        legendgroup=meas_name,
                        showlegend=meas_name not in legend_seen,
                    ),
                    row=row_idx,
                    col=col_idx,
                )
                legend_seen.add(meas_name)

                if y_lo is not None and y_hi is not None:
                    mask = np.isfinite(y_meas) & np.isfinite(y_lo) & np.isfinite(y_hi) & outside_mask(y_meas, y_lo, y_hi)
                    if np.any(mask):
                        outside_name = "Measured Outside Range"
                        fig.add_trace(
                            go.Scatter(
                                x=x[mask],
                                y=y_meas[mask],
                                mode="markers",
                                marker=dict(symbol="x", size=10, color=COLORS["outside"], line=dict(width=2)),
                                name=outside_name,
                                legendgroup=outside_name,
                                showlegend=outside_name not in legend_seen,
                            ),
                            row=row_idx,
                            col=col_idx,
                        )
                        legend_seen.add(outside_name)

            fig.update_xaxes(
                title_text=str(sweep_var),
                showgrid=True,
                gridcolor="rgba(0,0,0,0.15)",
                row=row_idx,
                col=col_idx,
            )
            fig.update_yaxes(
                title_text=str(out_var),
                showgrid=True,
                gridcolor="rgba(0,0,0,0.15)",
                exponentformat="SI",
                row=row_idx,
                col=col_idx,
            )

        fig.update_layout(
            template="plotly_white",
            title=dict(
                text=make_group_title(panel_df, bid, context_vars),
                x=0,
                xanchor="left",
                y=0.99,
                yanchor="top",
                pad=dict(b=14),
            ),
            height=max(420, 420 * nrows),
            width=1200 if ncols > 1 else 760,
            margin=dict(l=30, r=20, t=120, b=30),
            legend=dict(orientation="h", yanchor="bottom", y=1.10, xanchor="left", x=0),
            hovermode="x unified",
        )
        display(fig)




In [7]:
if "plot_grouped_blocks" not in globals():
    raise RuntimeError("Run the previous 'Plotting and Styles' cell first.")

current_df = None
current_input_vars = []
filter_widgets = {}
updating_filter_widgets = False

device_dropdown = widgets.Dropdown(description="Device:", layout=widgets.Layout(width="420px"))
simulation_dropdown = widgets.Dropdown(description="Simulation:", layout=widgets.Layout(width="420px"))
plot_button = widgets.Button(description="Generate Plot", button_style="primary", icon="line-chart")
columns_output = widgets.Output()
filters_container = widgets.VBox()
plot_output = widgets.Output()


def parse_vars_from_row0(df, col_name):
    if df is None or df.empty or col_name not in df.columns:
        return []
    raw = str(df[col_name].iloc[0])
    return [v.strip() for v in raw.split(",") if v.strip()]


def parse_output_vars(df):
    return parse_vars_from_row0(df, "output_vars")


def parse_input_vars(df):
    candidates = parse_vars_from_row0(df, "input_vars")
    return [v for v in candidates if v in df.columns]


def parse_sweep_var(df):
    if "sweep_var" not in df.columns:
        return ""
    non_na_sweep = df["sweep_var"].dropna()
    if non_na_sweep.empty:
        return ""
    return str(non_na_sweep.iloc[0]).strip()


def get_filter_columns(df):
    sweep_var = parse_sweep_var(df)
    output_vars = parse_output_vars(df)

    base_cols = [v for v in parse_input_vars(df) if v != sweep_var and v in df.columns]

    excluded = {
        "block_id",
        "block_index",
        "index",
        "input_data",
        "input_vars",
        "output_vars",
        "sweep_var",
        "ps",
        "pd",
        "as",
        "ad"
    }
    excluded.update({c for c in df.columns if c.lower().endswith("_index")})
    if sweep_var:
        excluded.add(sweep_var)

    excluded.update(output_vars)
    excluded.update({f"{out_var}_meas" for out_var in output_vars})
    for out_var in output_vars:
        sim_prefix = f"{out_var}_sim_"
        excluded.update({c for c in df.columns if c.startswith(sim_prefix)})

    excluded.update({c for c in df.columns if c.endswith("_meas") or "_sim_" in c})

    extra_cols = []
    for col in df.columns:
        if col in excluded or col in base_cols:
            continue
        if df[col].dropna().empty:
            continue
        if df[col].nunique(dropna=True) <= 1:
            continue
        extra_cols.append(col)

    return base_cols + extra_cols, sweep_var


def is_numeric_col(series):
    return pd.api.types.is_numeric_dtype(series)


def build_filter_dropdown_options(series):
    values = series.dropna().unique().tolist()
    if is_numeric_col(series):
        values = sorted(values)
    else:
        values = sorted(values, key=lambda x: str(x))

    options = [("All", ALL_VALUE)]
    for v in values:
        options.append((format_value(v), v))
    return options


def update_device_options():
    device_options = sorted([p.name for p in RESULTS_DIR.iterdir() if p.is_dir()])
    print(f"Found device folders: {device_options}")
    if not device_options:
        raise ValueError(f"No device folders found in {RESULTS_DIR}")

    device_dropdown.options = device_options
    preferred = None
    for dev in device_options:
        combined_dir = RESULTS_DIR / dev / "combined_results"
        if combined_dir.exists() and any(combined_dir.glob("*.csv")):
            preferred = dev
            break
    device_dropdown.value = preferred if preferred is not None else device_options[0]


def update_simulation_options(*_):
    combined_dir = RESULTS_DIR / device_dropdown.value / "combined_results"
    csv_files = sorted([p.name for p in combined_dir.glob("*.csv") if p.is_file()]) if combined_dir.exists() else []

    if csv_files:
        simulation_dropdown.options = csv_files
        simulation_dropdown.value = csv_files[0]
    else:
        simulation_dropdown.options = ["<no csv files found>"]
        simulation_dropdown.value = "<no csv files found>"


def build_input_filters(df):
    global filter_widgets, current_input_vars
    current_input_vars, _ = get_filter_columns(df)
    filter_widgets = {}

    if not current_input_vars:
        filters_container.children = [widgets.HTML("<b>No filter variables found after exclusions.</b>")]
        return

    dropdowns = []
    for var in current_input_vars:
        dd = widgets.Dropdown(
            options=build_filter_dropdown_options(df[var]),
            value=ALL_VALUE,
            description=f"{var}:",
            layout=widgets.Layout(width="260px")
        )
        dd.observe(on_filter_change, names="value")
        filter_widgets[var] = dd
        dropdowns.append(dd)

    grid = widgets.GridBox(
        children=dropdowns,
        layout=widgets.Layout(grid_template_columns="repeat(3, minmax(220px, 1fr))", grid_gap="8px 12px")
    )
    filters_container.children = [widgets.HTML("<b>Filters:</b>"), grid]


def apply_single_filter(df, var, selected):
    if selected == ALL_VALUE or var not in df.columns:
        return df

    if is_numeric_col(df[var]):
        col_values = pd.to_numeric(df[var], errors="coerce")
        return df[np.isclose(col_values, float(selected), rtol=1e-9, atol=1e-18)]
    return df[df[var] == selected]


def get_filtered_df(df, exclude_var=None):
    filtered = df.copy()
    for var, dd in filter_widgets.items():
        if exclude_var is not None and var == exclude_var:
            continue

        filtered = apply_single_filter(filtered, var, dd.value)
        if filtered.empty:
            break

    return filtered


def get_plot_count(df):
    if df is None or df.empty:
        return 0

    work = df.copy()
    if "sweep_var" in work.columns:
        sweep_values = [s for s in work["sweep_var"].dropna().astype(str).unique().tolist() if s]
        if not sweep_values:
            return 0
        sweep_var = sweep_values[0]
        work = work[work["sweep_var"].astype(str) == sweep_var]
        if work.empty:
            return 0

    if "block_id" not in work.columns:
        return 0
    return int(work["block_id"].nunique())


def refresh_filter_dropdowns():
    global updating_filter_widgets
    if current_df is None or current_df.empty or not filter_widgets:
        return

    updating_filter_widgets = True
    try:
        max_passes = max(2, len(filter_widgets) * 2)
        for _ in range(max_passes):
            changed = False
            current_values = {var: dd.value for var, dd in filter_widgets.items()}

            for var, dd in filter_widgets.items():
                constrained = get_filtered_df(current_df, exclude_var=var)
                options = build_filter_dropdown_options(constrained[var]) if var in constrained.columns else [("All", ALL_VALUE)]
                dd.options = options

                valid_values = {value for _, value in options}
                target_value = current_values.get(var, ALL_VALUE)
                if target_value not in valid_values:
                    target_value = ALL_VALUE

                if dd.value != target_value:
                    dd.value = target_value
                    changed = True

            if not changed:
                break
    finally:
        updating_filter_widgets = False


def on_filter_change(_):
    if updating_filter_widgets:
        return

    refresh_filter_dropdowns()
    plots_left = get_plot_count(get_filtered_df(current_df)) if current_df is not None else 0
    with plot_output:
        clear_output(wait=True)
        print(f"Filters updated. Matching plots: {plots_left}. Click 'Generate Plot' to render.")


def render_plot():
    with plot_output:
        clear_output(wait=True)

        if current_df is None or current_df.empty:
            print("No CSV loaded yet.")
            return

        filtered = get_filtered_df(current_df)
        if filtered.empty:
            print("No rows match selected filter values.")
            return

        output_vars = parse_output_vars(filtered)
        if not output_vars:
            print("No output_vars found in row[0].")
            return

        if "sweep_var" not in filtered.columns:
            print("Column 'sweep_var' is missing in CSV.")
            return

        sweep_values = [s for s in filtered["sweep_var"].dropna().astype(str).unique().tolist() if s]
        if not sweep_values:
            print("No valid sweep_var value found.")
            return

        sweep_var = sweep_values[0]
        filtered = filtered[filtered["sweep_var"].astype(str) == sweep_var].copy()
        if sweep_var not in filtered.columns:
            print(f"Sweep column '{sweep_var}' is not present in this CSV.")
            return

        plot_grouped_blocks(filtered, output_vars, sweep_var, current_input_vars)


def on_plot_button_clicked(_):
    plot_button.disabled = True
    prev_label = plot_button.description
    prev_icon = plot_button.icon
    plot_button.description = "Plotting..."
    plot_button.icon = "spinner"
    try:
        render_plot()
    except Exception as exc:
        with plot_output:
            clear_output(wait=True)
            print(f"Plot failed: {exc}")
    finally:
        plot_button.description = prev_label
        plot_button.icon = prev_icon
        plot_button.disabled = False


def load_selected_csv(*_):
    global current_df
    with columns_output:
        clear_output(wait=True)

        selected_sim = simulation_dropdown.value
        if not selected_sim or str(selected_sim).startswith("<"):
            current_df = None
            filters_container.children = [widgets.HTML("<b>No simulation CSV available for this device.</b>")]
            with plot_output:
                clear_output(wait=True)
                print("No data to plot.")
            return

        csv_path = RESULTS_DIR / device_dropdown.value / "combined_results" / selected_sim
        current_df = pd.read_csv(csv_path)

        print(f"Selected CSV: {csv_path}")

    build_input_filters(current_df)
    refresh_filter_dropdowns()
    with plot_output:
        clear_output(wait=True)
        plots_left = get_plot_count(get_filtered_df(current_df)) if current_df is not None else 0
        print(f"Filters updated. Matching plots: {plots_left}. Click 'Generate Plot' to render.")


device_dropdown.observe(update_simulation_options, names="value")
simulation_dropdown.observe(load_selected_csv, names="value")
plot_button.on_click(on_plot_button_clicked)

update_device_options()
update_simulation_options()
load_selected_csv()

display(widgets.VBox([
    widgets.HBox([device_dropdown, simulation_dropdown, plot_button]),
    columns_output,
    filters_container,
    widgets.HTML("<br>"),
    plot_output
]))






Found device folders: ['nmos_lv']
